# DINOv3 Backbone Verification

**Model**: `facebook/dinov3-vitl16-pretrain-lvd1689m` (and other sizes/datasets)  
**Arch**: ViT with RoPE positional encoding + 4 register tokens (ViT-S/B/L/H/7B sizes)  
**Embed dim**: 1024 (ViT-L) · 768 (ViT-B) · 384 (ViT-S) · 1280 (ViT-H) · larger for 7B  
**Input**: 224×224 px default; natively supports any resolution that is a multiple of patch size 16  
**Access**: HuggingFace, no auth  
**Zero-shot**: No — vision-only SSL

Paper: [DINOv3: Scaling Vision Self-Supervised Learning to 7B Parameters and 1.7B Images (Meta AI, 2025)](https://ai.meta.com/dinov3/)

Available variants (`facebook/dinov3-<arch>-pretrain-<dataset>`):

| Hub suffix | Arch | Params | Dataset |
|---|---|---|---|
| `vits16-pretrain-lvd1689m` | ViT-S/16 | 22M | LVD-1689M (web) |
| `vitb16-pretrain-lvd1689m` | ViT-B/16 | 86M | LVD-1689M |
| `vitl16-pretrain-lvd1689m` | ViT-L/16 | 300M | LVD-1689M |
| `vith16plus-pretrain-lvd1689m` | ViT-H+/16 | ~600M | LVD-1689M |
| `vit7b16-pretrain-lvd1689m` | ViT-7B/16 | 7B | LVD-1689M |
| `vit7b16-pretrain-sat493m` | ViT-7B/16 | 7B | SAT-493M (satellite) |
| `convnext-small-pretrain-lvd1689m` | ConvNeXt-S | — | LVD-1689M |

In [ ]:
# %pip install -q transformers torch torchvision Pillow

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Load model

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModel

# Pick a variant — ViT-B is practical; 7B requires ~28GB VRAM
HUB = "facebook/dinov3-vitb16-pretrain-lvd1689m"

processor = AutoImageProcessor.from_pretrained(HUB)
model     = AutoModel.from_pretrained(HUB)
model.eval()

print("Hub:", HUB)
print("Hidden size:", model.config.hidden_size)   # 1024 for ViT-L
print("Patch size: ", model.config.patch_size)    # 16
print("Register tokens:", getattr(model.config, "num_register_tokens", "N/A"))  # 4

## 2. Preprocess image

DINOv3 supports **any resolution** that is a multiple of the patch size (16),
allowing native-resolution inference without distortion.

In [ ]:
from PIL import Image
import numpy as np

def preprocess(path: str, size: int = 224) -> dict:
    """Load and resize to square multiple of 16. Default 224px."""
    img = Image.open(path).convert("RGB")
    # AutoImageProcessor resizes to its configured crop_size
    return processor(images=img, return_tensors="pt")

print("Processor crop size:", processor.crop_size)
print("Image mean:", processor.image_mean)
print("Image std: ", processor.image_std)

dummy_pil = Image.fromarray(np.ones((512, 512, 3), dtype=np.uint8) * 128)
inputs    = processor(images=dummy_pil, return_tensors="pt")
print("pixel_values:", inputs["pixel_values"].shape)   # [1, 3, 224, 224]

## 3. Extract embeddings

DINOv3 uses 4 register tokens. Token layout:
`[CLS] [reg1] [reg2] [reg3] [reg4] [patch_1] ... [patch_N]`

Use `pooler_output` (CLS) for global embedding.

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

seq = outputs.last_hidden_state                # [B, 1+4+N_patches, D]
cls_emb  = seq[:, 0]                           # [B, D] — CLS token
reg_emb  = seq[:, 1:5]                         # [B, 4, D] — register tokens (discard for embeddings)
patch_emb = seq[:, 5:]                         # [B, N_patches, D]
pooled   = outputs.pooler_output               # [B, D] — same as cls_emb
mean_emb = patch_emb.mean(dim=1)               # [B, D] — mean-pool patch tokens

n_patches = patch_emb.shape[1]
print(f"Total tokens: {seq.shape[1]}  (1 CLS + 4 reg + {n_patches} patches)")
print("CLS / pooled:", cls_emb.shape)          # [1, 1024]
print("patch tokens:", patch_emb.shape)        # [1, 196, 1024] for 224px/16
print("mean pool:   ", mean_emb.shape)

## Segmentation mode — spatial feature map

Reshape the patch sequence to a 2-D spatial grid. Always use `seq[:, 5:]` to skip
**1 CLS + 4 register tokens** — including register tokens in the spatial map corrupts
the positional layout. For ViT-L/16 at 224×224: **14×14** patch grid, 1024-dim.

In [ ]:
import math

# patch_emb from section 3: seq[:, 5:]  (skip 1 CLS + 4 register tokens)
# 224px / 16px-patches → 14×14 = 196 patches, D=1024 (ViT-L)
B, N, D = patch_emb.shape
H = W = int(math.sqrt(N))   # 14 for 224px / 16px-patches
assert H * W == N, f"Patch grid {N} is not a perfect square"

spatial = patch_emb.permute(0, 2, 1).reshape(B, D, H, W)
print(f"patch tokens:  {patch_emb.shape}")   # [1, 196, 1024]
print(f"spatial map:   {spatial.shape}")     # [1, 1024, 14, 14]

## 4. High-resolution inference (native aspect ratio)

In [ ]:
from torchvision import transforms

def make_hires_transform(size: int = 448):
    """Override processor at a larger resolution (must be multiple of 16)."""
    assert size % 16 == 0, f"size must be multiple of 16, got {size}"
    mean = processor.image_mean
    std  = processor.image_std
    return transforms.Compose([
        transforms.Resize(size),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

hires_transform = make_hires_transform(448)
hires_input = hires_transform(dummy_pil).unsqueeze(0)   # [1, 3, 448, 448]

with torch.no_grad():
    out_hires = model(pixel_values=hires_input)

print("Hi-res input:", hires_input.shape)
print("Hi-res tokens:", out_hires.last_hidden_state.shape)   # more patch tokens
print("Hi-res CLS:   ", out_hires.pooler_output.shape)

## 5. DINOv3 vs DINOv2 — embedding comparison

In [ ]:
import torch.nn.functional as F

# Load DINOv2 ViT-L for comparison
from transformers import AutoModel as AM, AutoImageProcessor as AIP

proc_v2 = AIP.from_pretrained("facebook/dinov2-large")
mdl_v2  = AM.from_pretrained("facebook/dinov2-large").eval()

inp_v2 = proc_v2(images=dummy_pil, return_tensors="pt")
with torch.no_grad():
    out_v2 = mdl_v2(**inp_v2)

emb_v2 = F.normalize(out_v2.pooler_output, dim=-1)
emb_v3 = F.normalize(outputs.pooler_output, dim=-1)

print("DINOv2 ViT-L embed:", emb_v2.shape)
print("DINOv3 ViT-L embed:", emb_v3.shape)
print("Cosine similarity (random image, should be low):",
      F.cosine_similarity(emb_v2, emb_v3).item())

## Notes for RadHarmony integration

```
model_call : lambda m, x: m(pixel_values=x).pooler_output   # CLS [B, D]
pool       : None
embed_dim  : 1024 (ViT-L) · 768 (ViT-B) · 384 (ViT-S)
transform  : AutoImageProcessor — resize/crop/normalize handled automatically
             Use EncoderPreprocessTransform.from_huggingface(keys=["img"], processor=processor)
```

**Register tokens**: DINOv3 emits 4 register tokens between CLS and patch tokens.
Always use `pooler_output` (CLS) or slice `seq[:, 5:]` for patches — do NOT
accidentally include register tokens in your mean pool.

**vs DINOv2**: DINOv3 was pretrained on a larger dataset (LVD-1689M vs 142M images)
with an improved recipe. Use the comparison cell above to check whether the gain
transfers to CXR embeddings.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Patch tokens: `model(pixel_values=x).last_hidden_state[:, 5:]` (skip CLS + 4 registers) → `[B, N, D]`
- Spatial map: reshape to `[B, 1024, 14, 14]` for ViT-L/16 at 224px (14²=196 patches)
- **Never use `[:, 1:]`** — that includes the 4 register tokens, which don't map to spatial positions

In [ ]:
CXR_PATH = "/path/to/cxr.jpg"

img = Image.open(CXR_PATH).convert("RGB")
inputs = processor(images=img, return_tensors="pt")

with torch.no_grad():
    out = model(**inputs)
    emb = out.pooler_output   # [1, D]

print("embedding:", emb.shape, "  norm:", emb.norm(dim=-1).item())

## Notes for RadHarmony integration

```
model_call : lambda m, x: m(pixel_values=x).pooler_output   # CLS [B, D]
pool       : None
embed_dim  : 1024 (ViT-L) · 768 (ViT-B) · 384 (ViT-S)
transform  : AutoImageProcessor — resize/crop/normalize handled automatically
             Use EncoderPreprocessTransform.from_huggingface(keys=["img"], processor=processor)
```

**Register tokens**: DINOv3 emits 4 register tokens between CLS and patch tokens.
Always use `pooler_output` (CLS) or slice `seq[:, 5:]` for patches — do NOT
accidentally include register tokens in your mean pool.

**vs DINOv2**: DINOv3 was pretrained on a larger dataset (LVD-1689M vs 142M images)
with an improved recipe. Use the comparison cell above to check whether the gain
transfers to CXR embeddings.